# Endoscapes Paper Example

Compare all seven methods for the three paper models on Figure 2's frame, video 184 / frame 36750. The cells below also identify cases where weighted rubric scoring succeeds and the baselines fail.


In [ ]:
from __future__ import annotations

import json
import random
from pathlib import Path
from typing import Dict

import numpy as np
from IPython.display import display, HTML, Image

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'rubrics/cvs_rubrics_v3.json').is_file())
OUTPUT_ROOT = REPO_ROOT / 'outputs' / 'rubric_oracle' / 'manifests' / 'test_dev'
RUBRICS_PATH = REPO_ROOT / 'rubrics' / 'cvs_rubrics_v3.json'
CRITS = ['c1', 'c2', 'c3']
CRIT_NAMES = {'c1': 'Two Structures', 'c2': 'Clear Triangle', 'c3': 'Detachment'}
THRESH = 0.5
SEED = 42
N_EXAMPLES = 5
PREFIX = 'endoscapes_val'
RUN = 'run001'

random.seed(SEED)
np.random.seed(SEED)

# ── Rubric weights (for Rubric(w) scoring) ──

def status_to_bin(status):
    return 1 if status == 'yes' else 0

rubrics_data = json.loads(RUBRICS_PATH.read_text())
rubric_items: Dict[str, Dict[str, object]] = {}
for criterion, block in rubrics_data.get('criteria', {}).items():
    for item in block.get('items', []):
        item_id = item.get('id')
        if item_id:
            rubric_items[item_id] = {'criterion': criterion,
                                     'weight': float(item.get('weight', 0.0))}

def weighted_score_from_rubrics(pred_rubrics: dict) -> dict:
    """Compute weighted score per criterion from Stage 1 yes/no labels."""
    totals, weights = {}, {}
    for item_id, status in pred_rubrics.items():
        meta = rubric_items.get(item_id)
        if not meta:
            continue
        c = str(meta['criterion']).lower()
        w = float(meta.get('weight', 0.0))
        totals[c] = totals.get(c, 0.0) + w * status_to_bin(status)
        weights[c] = weights.get(c, 0.0) + w
    return {c: (totals.get(c, 0.0) / weights[c] if weights.get(c, 0) else float('nan'))
            for c in CRITS}

print(f'Loaded {len(rubric_items)} rubric items with weights')

# ── Utilities ──

def iter_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def index_by_frame(rows: list[dict]) -> dict:
    """Index rows by (video_id, frame_id), keeping first occurrence."""
    by_frame = {}
    for row in rows:
        key = (row['video_id'], row['frame_id'])
        if key not in by_frame:
            by_frame[key] = row
    return by_frame


def get_binary(d: dict, thresh: float = 0.5) -> dict:
    result = {}
    for c in CRITS:
        v = d.get(c)
        if v is not None and not (isinstance(v, float) and np.isnan(v)):
            result[c] = 1 if float(v) > thresh else 0
    return result


def make_path(exp_label, model_id):
    """Construct output file path for Endoscapes test_dev."""
    fname = f'{PREFIX}__cvsrubricsv3__annv2__{exp_label}__seed13__{model_id}_{RUN}.jsonl'
    return OUTPUT_ROOT / fname


print('Setup done.')

In [ ]:
SPLIT = 'test_dev'

MODELS = [
    ('gpt-4.1-mini',              'GPT-4.1-mini'),
    ('claude-haiku-4-5-20251001', 'Claude Haiku 4.5'),
    ('claude-opus-4-5-20251101',  'Claude Opus 4.5'),
]

# (method_display_name, exp_label, score_source, border_color)
# score_source: 'direct' uses row['pred'], 'weighted' uses weighted_score_from_rubrics(row['rubric_labels_stage1'])
METHODS = [
    ('Direct',         f'direct__preset-direct__{SPLIT}__critdef',                            'direct',   '#3498db'),
    ('Direct+FS',      f'direct__preset-direct__{SPLIT}__critdef__fsv3__fs4__',               'direct',   '#2ecc71'),
    ('CoT-first',      f'direct__preset-direct__{SPLIT}__critdef__ratdf',                     'direct',   '#9b59b6'),
    ('CoT-first+FS',   f'direct__preset-direct__{SPLIT}__critdef__ratdf__fsv3__fs4__',        'direct',   '#27ae60'),
    ('Rubric(w)',       f'predicted_only__{SPLIT}__critdef__fsv3__fs4__',                      'weighted', '#e74c3c'),
    ('Self-Rubric',    f'self_rubric__{SPLIT}__critdef__rat1__s1short',                        'direct',   '#e67e22'),
    ('Self-Rubric+FS', f'self_rubric__{SPLIT}__critdef__rat1__s1short__fsv3__fs4__',           'direct',   '#f39c12'),
]

# Load all data: data[(model_id, method_name)] = {(vid, fid): row, ...}
data = {}
for model_id, model_name in MODELS:
    for method_name, exp_label, score_source, _ in METHODS:
        path = make_path(exp_label, model_id)
        key = (model_id, method_name)
        if not path.exists():
            print(f'WARNING: {model_name} / {method_name} not found: {path.name}')
            continue
        rows = iter_jsonl(path)
        data[key] = index_by_frame(rows)
        print(f'{model_name:20s} {method_name:16s}: {len(data[key])} frames')

print(f'\nLoaded {len(data)} experiment variants.')

# Find common frames across ALL loaded experiments
all_frame_sets = [set(d.keys()) for d in data.values()]
common_frames = set.intersection(*all_frame_sets) if all_frame_sets else set()
print(f'Common frames across all {len(data)} experiments: {len(common_frames)}')

In [ ]:
selected_frames = [('184', 36750)]
print(f'Selected {len(selected_frames)} frames:')
for vid, fid in selected_frames:
    # Use any loaded experiment to get GT
    first_key = next(iter(data))
    row = data[first_key][(vid, fid)]
    gt = row.get('gt', {})
    gt_bin = get_binary(gt)
    labels = ' '.join(f'{c.upper()}={gt_bin.get(c, "?")}' for c in CRITS)
    print(f'  {vid} / frame {fid}  GT: {labels}')

In [ ]:
def fmt_gt(gt: dict) -> str:
    parts = []
    for k in CRITS:
        v = gt.get(k)
        if v is None:
            parts.append(f'{k.upper()}=?')
        else:
            fv = float(v)
            label = 1 if fv > 0.5 else 0
            color = '#2ca02c' if label == 1 else '#d62728'
            parts.append(f'<span style="color:{color};font-weight:bold">{k.upper()}={fv:.2f}</span>')
    return ' &nbsp; '.join(parts)


def fmt_pred(pred: dict, gt: dict = None) -> str:
    if not pred:
        return '<i>no prediction</i>'
    parts = []
    for k in CRITS:
        v = pred.get(k)
        if v is None:
            parts.append(f'{k.upper()}=?')
        else:
            fv = float(v)
            pred_bin = 1 if fv > THRESH else 0
            gt_bin = 1 if gt and gt.get(k) is not None and float(gt[k]) > THRESH else 0
            if gt is not None:
                match = pred_bin == gt_bin
                icon = '&#10003;' if match else '&#10007;'
                color = '#2ca02c' if match else '#d62728'
            else:
                icon = ''
                color = '#333'
            parts.append(f'<span style="color:{color};font-weight:bold">{k.upper()}={fv:.2f} {icon}</span>')
    return ' &nbsp; '.join(parts)


def fmt_rubric_labels(labels: dict) -> str:
    if not labels:
        return ''
    color_map = {'yes': '#2ca02c', 'no': '#d62728', 'uncertain': '#ff7f0e'}
    rows_html = []
    for item_id in sorted(labels.keys()):
        ans = labels[item_id]
        color = color_map.get(str(ans).lower(), '#333')
        rows_html.append(
            f'<tr><td style="font-size:11px;padding:1px 4px">{item_id}</td>'
            f'<td style="color:{color};font-weight:bold;font-size:11px;padding:1px 4px">{ans}</td></tr>'
        )
    return f'<table style="border-collapse:collapse;margin:2px 0">{"".join(rows_html)}</table>'


def fmt_self_rubric_checklist(checklist: dict) -> str:
    if not checklist:
        return ''
    color_map = {'yes': '#2ca02c', 'no': '#d62728', 'uncertain': '#ff7f0e'}
    rows_html = []
    for criterion in ('C1', 'C2', 'C3'):
        items = checklist.get(criterion, [])
        if not items:
            continue
        rows_html.append(
            f'<tr><td colspan="2" style="font-size:11px;padding:3px 4px 1px;font-weight:bold;'
            f'border-top:1px solid #ddd">{criterion}</td></tr>'
        )
        for item in items:
            q = item.get('question', '')
            a = str(item.get('answer', 'uncertain')).lower()
            color = color_map.get(a, '#333')
            rows_html.append(
                f'<tr><td style="font-size:10px;padding:1px 4px 1px 12px;max-width:500px">{q}</td>'
                f'<td style="color:{color};font-weight:bold;font-size:11px;padding:1px 4px;white-space:nowrap">{a}</td></tr>'
            )
    return f'<table style="border-collapse:collapse;margin:2px 0">{"".join(rows_html)}</table>'


def show_method_response(row: dict, method_label: str, gt_raw: dict,
                         score_source: str, border_color: str):
    """Show a single method's prediction + reasoning for a frame."""
    html = f'<div style="border-left:3px solid {border_color};padding:5px 10px;margin:5px 0">'
    html += f'<b>{method_label}</b><br>'

    if score_source == 'weighted':
        # Rubric(w): show weighted score from Stage 1 labels
        rubric_labels = row.get('rubric_labels_stage1', {}) or {}
        if rubric_labels:
            ws = weighted_score_from_rubrics(rubric_labels)
            html += f'<b>Pred (Rubric(w)):</b> {fmt_pred(ws, gt_raw)}<br>'
        else:
            html += '<b>Pred (Rubric(w)):</b> <i>no rubric labels</i><br>'
        # Also show direct Stage 2 pred for comparison
        pred = row.get('pred', {})
        if pred:
            html += f'<b>Pred (Stage 2):</b> {fmt_pred(pred, gt_raw)}<br>'
    else:
        pred = row.get('pred', {})
        html += f'<b>Pred:</b> {fmt_pred(pred, gt_raw)}<br>'

    # Stage 1 rubric labels (for rubric-based methods)
    rubric_labels = row.get('rubric_labels_stage1', {}) or {}
    if rubric_labels:
        html += '<details><summary><b>Rubric Labels (Stage 1)</b></summary>'
        html += fmt_rubric_labels(rubric_labels)
        html += '</details>'

    # Self-rubric checklist
    checklist = row.get('checklist', {}) or {}
    if checklist:
        html += '<details><summary><b>Self-Rubric Checklist</b></summary>'
        html += fmt_self_rubric_checklist(checklist)
        html += '</details>'

    # Stage 1 raw response
    s1_raw = row.get('stage1_raw_response', '')
    if s1_raw:
        s1_trunc = s1_raw[:800] + ('...' if len(s1_raw) > 800 else '')
        html += f'<details><summary>Stage 1 Response ({len(s1_raw)} chars)</summary>'
        html += f'<pre style="font-size:11px;white-space:pre-wrap;max-height:200px;overflow-y:auto">{s1_trunc}</pre></details>'

    # Final raw response
    raw = row.get('raw_response', '')
    if raw:
        truncated = raw[:800] + ('...' if len(raw) > 800 else '')
        html += f'<details><summary>Final Response ({len(raw)} chars)</summary>'
        html += f'<pre style="font-size:11px;white-space:pre-wrap;max-height:200px;overflow-y:auto">{truncated}</pre></details>'

    html += '</div>'
    display(HTML(html))


def show_frame(fkey, idx):
    """Display a full diagnostic view for one frame across all models and methods."""
    vid, fid = fkey
    first_key = next(iter(data))
    row0 = data[first_key][fkey]
    gt_raw = row0.get('gt', {})
    img_path = str(REPO_ROOT / 'data/endoscapes/test' / f"{row0['video_id']}_{row0['frame_id']}.jpg")

    display(HTML(f'<hr style="border:2px solid #333"><h2>Example {idx+1}: video {vid} / frame {fid}</h2>'))
    display(HTML(f'<b>GT:</b> {fmt_gt(gt_raw)}'))

    if Path(img_path).exists():
        display(Image(filename=img_path, width=500))
    else:
        display(HTML(f'<i>Image not found: {img_path}</i>'))

    for model_id, model_name in MODELS:
        display(HTML(f'<h3 style="margin-top:15px;border-bottom:1px solid #ccc">{model_name}</h3>'))
        for method_name, exp_label, score_source, border_color in METHODS:
            key = (model_id, method_name)
            if key not in data or fkey not in data[key]:
                display(HTML(f'<div style="border-left:3px solid #ccc;padding:5px 10px;margin:5px 0">'
                             f'<b>{method_name}</b>: <i>no data</i></div>'))
                continue
            show_method_response(
                data[key][fkey],
                method_name,
                gt_raw,
                score_source,
                border_color,
            )


# ── Show selected frames ──
for idx, fkey in enumerate(selected_frames):
    show_frame(fkey, idx)

print(f'\nDone — {len(selected_frames)} frames displayed.')

## Rubric(w) Wins — Cases where Rubric(w) is correct but all other methods are wrong

For each model, find (frame, criterion) pairs where Rubric(w) predicts the correct
binary label but every other method (Direct, Direct+FS, CoT-first, CoT-first+FS,
Self-Rubric, Self-Rubric+FS) gets it wrong. Then pick 5 to visualise.

In [ ]:
OTHER_METHODS = [m for m in METHODS if m[0] != 'Rubric(w)']
RUBRIC_W_METHOD = next(m for m in METHODS if m[0] == 'Rubric(w)')

def get_pred_binary(fkey, model_id, method_tuple):
    """Get binary predictions for a (frame, model, method) combination."""
    method_name, exp_label, score_source, _ = method_tuple
    key = (model_id, method_name)
    if key not in data or fkey not in data[key]:
        return {}
    row = data[key][fkey]
    if score_source == 'weighted':
        rubric_labels = row.get('rubric_labels_stage1', {}) or {}
        if not rubric_labels:
            return {}
        ws = weighted_score_from_rubrics(rubric_labels)
        return get_binary(ws)
    else:
        pred = row.get('pred', {}) or {}
        return get_binary(pred)

# Find (model, frame, criterion) where Rubric(w) correct but ALL others wrong
rubric_w_wins = []  # (model_id, model_name, fkey, criterion)

for model_id, model_name in MODELS:
    for fkey in common_frames:
        first_key = next(iter(data))
        gt_raw = data[first_key][fkey].get('gt', {})
        gt = get_binary(gt_raw)
        if len(gt) < 3:
            continue

        rw_pred = get_pred_binary(fkey, model_id, RUBRIC_W_METHOD)

        for c in CRITS:
            if c not in rw_pred or c not in gt:
                continue
            if rw_pred[c] != gt[c]:
                continue  # Rubric(w) is wrong here

            # Check all other methods are wrong on this criterion
            all_others_wrong = True
            for other_method in OTHER_METHODS:
                other_pred = get_pred_binary(fkey, model_id, other_method)
                if c in other_pred and other_pred[c] == gt[c]:
                    all_others_wrong = False
                    break

            if all_others_wrong:
                rubric_w_wins.append((model_id, model_name, fkey, c))

print(f'Total (model, frame, criterion) where Rubric(w) wins uniquely: {len(rubric_w_wins)}')
print()

# Breakdown by model and criterion
from collections import Counter
for model_id, model_name in MODELS:
    hits = [(fkey, c) for mid, _, fkey, c in rubric_w_wins if mid == model_id]
    crit_counts = Counter(c for _, c in hits)
    frame_count = len(set(fkey for fkey, _ in hits))
    print(f'{model_name}: {len(hits)} criterion-hits across {frame_count} frames  '
          f'(C1={crit_counts.get("c1",0)}, C2={crit_counts.get("c2",0)}, C3={crit_counts.get("c3",0)})')

# Select up to 5 unique frames, preferring frames with multiple criterion wins
# Group by (model, frame), pick the model+frame combos with most criteria
from collections import defaultdict
by_model_frame = defaultdict(list)
for model_id, model_name, fkey, c in rubric_w_wins:
    by_model_frame[(model_id, model_name, fkey)].append(c)

sorted_candidates = sorted(by_model_frame.items(), key=lambda x: (-len(x[1]), x[0]))

N_RUBRIC_W_EXAMPLES = 5
selected_rw = sorted_candidates[:N_RUBRIC_W_EXAMPLES]

print(f'\nSelected {len(selected_rw)} frames to visualise:')
for (model_id, model_name, fkey), crits in selected_rw:
    vid, fid = fkey
    print(f'  {model_name}: {vid}/{fid}  criteria: {", ".join(c.upper() for c in crits)}')

In [ ]:
def show_frame_single_model(fkey, model_id, model_name, win_criteria, idx):
    """Display a frame showing all methods for a single model, highlighting winning criteria."""
    vid, fid = fkey
    first_key = next(iter(data))
    row0 = data[first_key][fkey]
    gt_raw = row0.get('gt', {})
    img_path = str(REPO_ROOT / 'data/endoscapes/test' / f"{row0['video_id']}_{row0['frame_id']}.jpg")

    crit_str = ', '.join(c.upper() for c in win_criteria)
    display(HTML(
        f'<hr style="border:2px solid #333">'
        f'<h2>Rubric(w) Win #{idx+1}: video {vid} / frame {fid}</h2>'
        f'<b>Model:</b> {model_name} &nbsp; | &nbsp; '
        f'<b>Rubric(w) uniquely correct on:</b> <span style="color:#e74c3c;font-weight:bold">{crit_str}</span>'
    ))
    display(HTML(f'<b>GT:</b> {fmt_gt(gt_raw)}'))

    if Path(img_path).exists():
        display(Image(filename=img_path, width=500))
    else:
        display(HTML(f'<i>Image not found: {img_path}</i>'))

    display(HTML(f'<h3>{model_name}</h3>'))
    for method_name, exp_label, score_source, border_color in METHODS:
        key = (model_id, method_name)
        if key not in data or fkey not in data[key]:
            display(HTML(f'<div style="border-left:3px solid #ccc;padding:5px 10px;margin:5px 0">'
                         f'<b>{method_name}</b>: <i>no data</i></div>'))
            continue
        show_method_response(data[key][fkey], method_name, gt_raw, score_source, border_color)


for idx, ((model_id, model_name, fkey), crits) in enumerate(selected_rw):
    show_frame_single_model(fkey, model_id, model_name, crits, idx)

print(f'\nDone — {len(selected_rw)} Rubric(w) win examples displayed.')